### I recommend you make a copy of the notebook and date it to keep the default one on handby 
### this also makes it easier to track the changes you make dependning on the experiments you're looking at 
#### note that this isn't 100% accurate and will sometimes give you channels for which cells were in the channels but could have left in the middle of the timelapse

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import warnings
import networkx as nx
import tifffile
import scipy.ndimage as ndi
import json

### Here you should put in your own input paths, the first two are just exmple formats of where everything should be saved
### Make sure that the CSV FOV # matches the FOV # of the tif file you're analyzing 

In [ ]:
# example of how to write out your paths 
csv_input_path_example = './EXP_NAME_DIR/hyperstacked/drift_corrected/rotated/mm_channelsFOVXYZ.csv"20250815_GNN_train_cell_lineage.ipynb'
rotated_tiff_path_example = './EXP_NAME_DIR/hyperstacked/drift_corrected/rotated/rotated_drift_cor_DuMM_xyFOV.tif'

In [ ]:
# replace these with your paths 
csv_path = "/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/image_analysis_testing/pos_list_test/hyperstacked/drift_corrected/rotated/mm_channelsFOV021.csv"
rotated_tif_path = "/Users/adrianjuarez/Documents/Covert_lab/Projects/Operon/image_analysis_testing/pos_list_test/hyperstacked/drift_corrected/rotated/rotated_drift_cor_DuMM_xy021.tif"

In [ ]:
# loads images into ndarrays and csv into pandas dataframe
trench_ids = pd.read_csv(csv_path)
image = tifffile.imread(rotated_tif_path)
phase_stack = image[:,0,:,:]
fluor_stack = image[:,1,:,:]


In [ ]:
# bottom and top pixels of image mess with normalization and averages
y_length = phase_stack.shape[1]  
y_start = int(0.1 * y_length)  
y_end = int(0.9 * y_length)  
# drift correct crop can also impact edge x pixels

x_length = phase_stack.shape[2]  
x_start = int(0.1 * y_length) 
x_end = int(0.9 * y_length)  

#normalizing using values from a cropped tiff file 
cropped_min_val = np.min(phase_stack[:, y_start:y_end, x_start:x_end])
cropped_max_val = np.max(phase_stack[:, y_start:y_end, x_start:x_end])
if cropped_max_val > cropped_min_val:
    normalized_phase_stack = (phase_stack - cropped_min_val) / (cropped_max_val - cropped_min_val)
else:
    print("All values in phase_stack are the same. Normalization will not change the data.")

In [ ]:
average_intensity = []  

for idx, row in trench_ids.iterrows():
    trench = row[0]  
    x = row[1]     
    
    # check that x pixel is inside the image 
    if 0 <= x < normalized_phase_stack.shape[2]:
        intensity_values = normalized_phase_stack[:, :, x+6] # pixel saved is slightly left of middle of channel
        intensity_values_middle = intensity_values[:, y_start:y_end]
        avg_int = np.min(intensity_values_middle, axis=0)  # find the minimum of each timepoint 
        
        final_avg_intensity = np.mean(avg_int)  # Further average over y dimension
        average_intensity.append((trench, final_avg_intensity))  # Store trench and avg intensity
    else:
        average_intensity.append((trench, None))  # Append None for out-of-bounds trench

# Convert results to a DataFrame
results_df = pd.DataFrame(average_intensity, columns=['trench_id', 'avg_int'])  # Store trench_id instead of channel_id

# Step 4: Determine empty and non-empty trenches based on a threshold
threshold = 0.25  # Example threshold (adjust as necessary)

results_df['has_cells'] = results_df['avg_int'].apply(lambda x: x is None or x < threshold)

#Display results
print(results_df)

In [ ]:
filtered_results = results_df[results_df['avg_int'].notna()]
plt.scatter(results_df['trench_id'], filtered_results['avg_int'])

plt.title('Trench ID vs Average Intensity')
plt.xlabel('Trench ID')
plt.ylabel('Average Intensity')
plt.grid(True)

### Here the print output should be in the format needed to run 02_background_subtraction
### you will need to pick out an empty trench yourself but you can see the list above for any example 
function 2 only does 1 FOV at a time so no need to try and find everything at once 

In [ ]:
cell_trenches = results_df[results_df['has_cells']]
anna_peak_ids = " ".join(f"'{int(trench_id)}'" for trench_id in cell_trenches['trench_id'])
print(anna_peak_ids)